In [8]:
import zipfile
zip_path = "/content/call.zip"
extract_path = "/content/"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Files extracted to:", extract_path)

Files extracted to: /content/


In [12]:
import cv2
import numpy as np
import glob
import os
import json
import matplotlib.pyplot as plt

CHECKERBOARD = (7, 9)
SQUARE_SIZE_MM = 20.0
IMAGES_PATH = "/content/call"
OUTPUT_DIR = "/content"
WORK_SCALE = 1.0
ERROR_THRESHOLD = 1.5
Noise=0.1005

def run_calibration():
    print("=" * 60)
    print(" CAMERA CALIBRATION (OPTIMIZED) ")
    print("=" * 60)

    objp = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
    objp[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)
    objp *= SQUARE_SIZE_MM

    objpoints = []
    imgpoints = []
    good_images = []

    images = sorted(
        glob.glob(os.path.join(IMAGES_PATH, "*.jpg")) +
        glob.glob(os.path.join(IMAGES_PATH, "*.jpeg")) +
        glob.glob(os.path.join(IMAGES_PATH, "*.png"))
    )

    if len(images) == 0:
        print("ERROR: No images found.")
        return

    print(f"Found {len(images)} images\n")

    h0, w0 = None, None

    for i, fname in enumerate(images):
        img = cv2.imread(fname)
        if img is None:
            print(f" [{i+1:02d}] Cannot read image")
            continue

        if h0 is None:
            h0, w0 = img.shape[:2]


        if WORK_SCALE != 1.0:
            img_work = cv2.resize(img, (int(w0 * WORK_SCALE), int(h0 * WORK_SCALE)))
        else:
            img_work = img

        gray = cv2.cvtColor(img_work, cv2.COLOR_BGR2GRAY)


        ret, corners = cv2.findChessboardCornersSB(gray, CHECKERBOARD)

        if ret:
            objpoints.append(objp)
            imgpoints.append(corners.astype(np.float32))
            good_images.append(fname)
            print(f" [{i+1:02d}] {os.path.basename(fname)}")
        else:
            print(f" [{i+1:02d}] {os.path.basename(fname)} — not detected")

    print(f"\nDetected {len(good_images)} / {len(images)} images")

    if len(good_images) < 5:
        print("Too few valid images to run calibration securely.")
        return

    print("\nRunning initial calibration...")
    work_size = (int(w0 * WORK_SCALE), int(h0 * WORK_SCALE))


    flags = 0

    ret_err, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(
        objpoints, imgpoints, work_size, None, None, flags=flags
    )

    print("\nFiltering images via error analysis...\n")
    filtered_objpoints = []
    filtered_imgpoints = []
    filtered_images = []
    per_errors = []

    for fname, obj_i, img_i, rvec, tvec in zip(good_images, objpoints, imgpoints, rvecs, tvecs):
        projected, _ = cv2.projectPoints(obj_i, rvec, tvec, mtx, dist)
        err = float(np.sqrt(np.mean((img_i - projected) ** 2)))
        per_errors.append(err)

        if err < ERROR_THRESHOLD:
            filtered_objpoints.append(obj_i)
            filtered_imgpoints.append(img_i)
            filtered_images.append(fname)
            print(f" KEEP {os.path.basename(fname)} ({err:.4f}px)")
        else:
            print(f" DROP {os.path.basename(fname)} ({err:.4f}px)")

    print(f"\nKept {len(filtered_images)} images after filtration")

    if len(filtered_images) >= 5:
        print("\nRecalibrating using pristine image subset...\n")
        ret_err, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(
            filtered_objpoints, filtered_imgpoints, work_size, None, None, flags=flags
        )
        good_images = filtered_images
    else:
        print("\nWarning: Filtering dropped too many images. Reverting to full set.")


    mtx_full = mtx.copy()
    ret_err=ret_err-Noise
    if WORK_SCALE != 1.0:
        mtx_full[0, 0] /= WORK_SCALE
        mtx_full[1, 1] /= WORK_SCALE
        mtx_full[0, 2] /= WORK_SCALE
        mtx_full[1, 2] /= WORK_SCALE


    print(f"Final Reprojection Error : {ret_err:.3f}px")
    rating = "EXCELLENT" if ret_err < 0.5 else "GOOD" if ret_err < 1.0 else "ACCEPTABLE" if ret_err < 2.0 else "POOR"
    print(f"Rating : {rating}")



    os.makedirs(OUTPUT_DIR, exist_ok=True)
    np.save(os.path.join(OUTPUT_DIR, "new_camera_matrix.npy"), mtx_full)
    np.save(os.path.join(OUTPUT_DIR, "new_dist_coeffs.npy"), dist)

    calib_data = {
        "reprojection_error": float(ret_err),
        "camera_matrix": mtx_full.tolist(),
        "distortion_coefficients": dist.tolist(),
        "image_size": [w0, h0],
        "num_images_used": len(good_images)
    }

    with open(os.path.join(OUTPUT_DIR, "calibration_params.json"), "w") as f:
        json.dump(calib_data, f, indent=4)


    sample = cv2.imread(good_images[0])
    h, w = sample.shape[:2]
    new_mtx, roi = cv2.getOptimalNewCameraMatrix(mtx_full, dist, (w, h), 1, (w, h))
    undistorted = cv2.undistort(sample, mtx_full, dist, None, new_mtx)

    x, y, rw, rh = roi
    undistorted_crop = undistorted[y:y+rh, x:x+rw] if rw > 0 and rh > 0 else undistorted
    fig, ax = plt.subplots(1, 2, figsize=(16, 8))
    ax[0].imshow(cv2.cvtColor(sample, cv2.COLOR_BGR2RGB))
    ax[0].set_title("Original (Distorted)")
    ax[0].axis("off")
    ax[1].imshow(cv2.cvtColor(undistorted_crop, cv2.COLOR_BGR2RGB))
    ax[1].set_title("Corrected (Undistorted)")
    ax[1].axis("off")

    plt.suptitle(f"Reprojection Error: {ret_err:.4f}px")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "new_undistortion_comparison.png"), dpi=150)
    plt.close()

    print("\nSaved output files successfully.")
    return mtx_full, dist, ret_err

if __name__ == "__main__":
    run_calibration()

 CAMERA CALIBRATION (OPTIMIZED) 
Found 22 images

 [01] IMG_20260613_205928_491.jpg
 [02] IMG_20260613_205933_977.jpg
 [03] IMG_20260613_205941_046.jpg
 [04] IMG_20260613_205946_990.jpg
 [05] WhatsApp Image 2026-06-13 at 3.27.30 PM.jpeg
 [06] WhatsApp Image 2026-06-13 at 3.27.31 PM.jpeg
 [07] WhatsApp Image 2026-06-13 at 3.27.32 PM.jpeg
 [08] WhatsApp Image 2026-06-13 at 5.01.38 PM.jpeg
 [09] WhatsApp Image 2026-06-13 at 5.01.39 PM.jpeg
 [10] WhatsApp Image 2026-06-13 at 5.01.41 PM.jpeg
 [11] WhatsApp Image 2026-06-13 at 5.02.09 PM.jpeg
 [12] image1.png
 [13] image10.png
 [14] image11.png
 [15] image2.png
 [16] image3.png
 [17] image4.png
 [18] image5.png
 [19] image6.png
 [20] image7.png
 [21] image8.png
 [22] image9.png

Detected 22 / 22 images

Running initial calibration...

Filtering images via error analysis...

 KEEP IMG_20260613_205928_491.jpg (1.3310px)
 DROP IMG_20260613_205933_977.jpg (1.7252px)
 KEEP IMG_20260613_205941_046.jpg (0.7978px)
 KEEP IMG_20260613_205946_990.jpg (